# Model Training

## Objectives
- Prepare image datasets for training
- Build a Convolutional Neural Network (CNN)
- Train the model to classify cherry leaves
- Apply techniques to prevent overfitting

## Inputs
- Training and validation datasets

## Outputs
- Trained CNN model
- Training history (accuracy and loss)
- Saved model file

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

In [3]:
import splitfolders

input_folder = "../inputs/cherry-leaves"
output_folder = "../outputs/datasets"

splitfolders.ratio(
    input_folder,
    output=output_folder,
    seed=42,
    ratio=(.7, .2, .1)
)

Copying files: 4208 files [00:01, 3223.05 files/s]


In [4]:
train_path = "../outputs/datasets/train"
val_path = "../outputs/datasets/val"
test_path = "../outputs/datasets/test"

In [5]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
IMG_SIZE = (100, 100)
BATCH_SIZE = 20

In [8]:
train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 2944 images belonging to 2 classes.
Found 840 images belonging to 2 classes.


In [9]:
model = models.Sequential([
    
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(100,100,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # prevent overfitting
    
    layers.Dense(1, activation='sigmoid')
])

In [10]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [11]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [12]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
148/148 [==============================] - 6s 42ms/step - loss: 0.2719 - accuracy: 0.8723 - val_loss: 0.1004 - val_accuracy: 0.9607
Epoch 2/20
148/148 [==============================] - 6s 40ms/step - loss: 0.0959 - accuracy: 0.9701 - val_loss: 0.0401 - val_accuracy: 0.9881
Epoch 3/20
148/148 [==============================] - 6s 42ms/step - loss: 0.0380 - accuracy: 0.9857 - val_loss: 0.0020 - val_accuracy: 1.0000
Epoch 4/20
148/148 [==============================] - 6s 42ms/step - loss: 0.0206 - accuracy: 0.9935 - val_loss: 0.0018 - val_accuracy: 1.0000
Epoch 5/20
148/148 [==============================] - 6s 43ms/step - loss: 0.0270 - accuracy: 0.9925 - val_loss: 0.0258 - val_accuracy: 0.9881
Epoch 6/20
148/148 [==============================] - 7s 46ms/step - loss: 0.0256 - accuracy: 0.9912 - val_loss: 0.0052 - val_accuracy: 0.9988
Epoch 7/20
148/148 [==============================] - 6s 43ms/step - loss: 0.0143 - accuracy: 0.9959 - val_loss: 0.0033 - val_accuracy: 1.0000

In [13]:
model.save("../outputs/models/mildew_detector.h5")

/Users/gregphillips/Library/CloudStorage/OneDrive-GregPhillips/Code institute/VS Code Projects/cherry-leaves-mildew-detection/venv/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [14]:
import pandas as pd

history_df = pd.DataFrame(history.history)
history_df.to_csv("../outputs/models/training_history.csv", index=False)